# LLM 정렬 코드 (실습용, Top-3 생성)

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GPT_MODEL      = os.getenv("GPT_MODEL", "gpt-4o-mini")

print("API 키 로드 완료")
print(f"GPT 모델: {GPT_MODEL}")


API 키 로드 완료
GPT 모델: gpt-4o-mini


In [4]:
import json
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate

In [5]:
# =========================
# 설정
# =========================
JOB_PATH = "datas/processed/job_postings_processed.csv"
NCS_PATH = "datas/processed/ncs_roles_processed.csv"
OUT_PATH = "outputs/llm_top3.csv"

MODEL_NAME = GPT_MODEL   # 사용 가능한 모델로 바꾸셔도 됩니다
TEMPERATURE = 0

# =========================
# 데이터 로드
# =========================
job = pd.read_csv(JOB_PATH, dtype={"job_posting_id": "string"})
ncs = pd.read_csv(NCS_PATH)

# 정리
job["job_posting_id"] = job["job_posting_id"].astype(str).str.strip()
job["text"] = job["text"].fillna("").astype(str)

ncs["job_role_name"] = ncs["job_role_name"].astype(str).str.strip()
# role_text가 없고 job_role_desc로 저장하셨을 수도 있어 안전 처리
if "role_text" not in ncs.columns:
    if "job_role_desc" in ncs.columns:
        ncs["role_text"] = ncs["job_role_desc"].fillna("").astype(str)
    else:
        raise ValueError("NCS 파일에 'role_text' 또는 'job_role_desc' 컬럼이 필요합니다.")
else:
    ncs["role_text"] = ncs["role_text"].fillna("").astype(str)

# NCS 목록 텍스트(너무 길면 LLM이 흔들리므로 120~200자 정도만)
ncs_list_text = "\n".join(
    [f"- {row['job_role_name']}: {row['role_text'][:150]}"
     for _, row in ncs.iterrows()]
)

# =========================
# LLM + 프롬프트
# =========================
llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=(
        "당신은 채용공고 텍스트를 NCS 세분류에 정렬하는 전문가입니다. "
        "반드시 지정된 JSON만 출력하고, 다른 설명을 덧붙이지 마세요."
    )),
    ("human", """
아래 채용공고를 읽고, [NCS 세분류 목록] 중 가장 적절한 3개를 순서대로 선택하세요.

규칙:
- 출력은 반드시 JSON 1개만 출력
- 값은 [NCS 세분류 목록]에 존재하는 '세분류명' 그대로 사용
- 형식:
{{
  "top1": "...",
  "top2": "...",
  "top3": "..."
}}

[채용공고]
{job_text}

[NCS 세분류 목록]
{ncs_list}
""")
])

# =========================
# 실행
# =========================
rows = []
MAX_JOB_CHARS = 2000  # 토큰 폭주 방지용

for _, r in job.iterrows():
    post_id = r["job_posting_id"]
    job_text = r["text"][:MAX_JOB_CHARS]

    msg = prompt.invoke({"job_text": job_text, "ncs_list": ncs_list_text})
    
    resp = llm.invoke(msg)
    content = (resp.content or "").strip()

    # JSON만 파싱
    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        # 혹시 코드블록 ```json ... ```로 감싸면 벗겨서 재시도
        cleaned = content.replace("```json", "").replace("```", "").strip()
        try:
            result = json.loads(cleaned)
        except json.JSONDecodeError:
            print(f"[WARN] JSON 파싱 실패: {post_id}")
            continue

    for i, key in enumerate(["top1", "top2", "top3"], start=1):
        rows.append({
            "job_posting_id": post_id,
            "rank": i,
            "job_role_name": str(result.get(key, "")).strip()
        })

out = pd.DataFrame(rows)
out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"[OK] saved: {OUT_PATH} | rows={len(out)} (= {len(job)} x 3, 일부 실패 제외)")

[OK] saved: outputs/llm_top3.csv | rows=324 (= 108 x 3, 일부 실패 제외)


## SBERT vs LLM Top-1/Top-3 + McNemar를 자동으로 계산

In [6]:
import pandas as pd
from math import comb

# =========================
# 경로 설정
# =========================
GOLD_PATH = "datas/processed/gold_final_20.csv"          # job_posting_id, gold_final
SBERT_PATH = "outputs/sbert_pilot_top3.csv"  # job_posting_id, rank, job_role_name
LLM_PATH  = "outputs/llm_top3.csv"       # job_posting_id, rank, job_role_name

TOP_K = 3

# =========================
# 유틸
# =========================
def load_gold(path: str) -> pd.DataFrame:
    gold = pd.read_csv(path, dtype={"job_posting_id": "string"})
    if not {"job_posting_id", "gold_final"}.issubset(gold.columns):
        raise ValueError("Gold 파일에 'job_posting_id', 'gold_final' 컬럼이 필요합니다.")
    gold["job_posting_id"] = gold["job_posting_id"].astype(str).str.strip()
    gold["gold_final"] = gold["gold_final"].astype(str).str.strip()
    return gold[["job_posting_id", "gold_final"]].drop_duplicates("job_posting_id")

def load_pred_topk(path: str, label_col="job_role_name", topk=3):
    df = pd.read_csv(path, dtype={"job_posting_id": "string"})
    need = {"job_posting_id", "rank", label_col}
    if not need.issubset(df.columns):
        raise ValueError(f"예측 파일에 필수 컬럼이 없습니다. 필요={need}, 현재={set(df.columns)}")

    df["job_posting_id"] = df["job_posting_id"].astype(str).str.strip()
    df["rank"] = df["rank"].astype(str).str.strip()
    df[label_col] = df[label_col].astype(str).str.strip()

    valid_ranks = {str(i) for i in range(1, topk + 1)} | {f"{i}.0" for i in range(1, topk + 1)}
    df = df[df["rank"].isin(valid_ranks)].copy()

    # Top-1
    top1 = df[df["rank"].isin(["1", "1.0"])][["job_posting_id", label_col]].rename(
        columns={label_col: "pred_top1"}
    )

    # Top-k list
    topk_list = (
        df.groupby("job_posting_id")[label_col]
        .apply(list)
        .reset_index()
        .rename(columns={label_col: "pred_topk_list"})
    )

    return top1, topk_list

def eval_top1_topk(gold_df: pd.DataFrame, top1_df: pd.DataFrame, topk_df: pd.DataFrame):
    e1 = gold_df.merge(top1_df, on="job_posting_id", how="inner")
    if len(e1) == 0:
        raise ValueError("Top-1 평가: Gold와 예측 간 매칭된 job_posting_id가 없습니다.")
    e1["correct_top1"] = (e1["gold_final"] == e1["pred_top1"])
    top1_acc = e1["correct_top1"].mean()

    ek = gold_df.merge(topk_df, on="job_posting_id", how="inner")
    if len(ek) == 0:
        raise ValueError("Top-k 평가: Gold와 예측 간 매칭된 job_posting_id가 없습니다.")
    ek["hit_topk"] = ek.apply(lambda r: r["gold_final"] in r["pred_topk_list"], axis=1)
    topk_acc = ek["hit_topk"].mean()
    return e1, ek, top1_acc, topk_acc

def mcnemar_exact_p(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    p = 0.0
    for i in range(0, k + 1):
        p += comb(n, i) * (0.5 ** n)
    p = 2 * p
    return min(1.0, p)

# =========================
# 1) 로드
# =========================
gold = load_gold(GOLD_PATH)

sbert_top1, sbert_top3 = load_pred_topk(SBERT_PATH, topk=TOP_K)
llm_top1, llm_top3     = load_pred_topk(LLM_PATH, topk=TOP_K)

# =========================
# 2) Top-1 / Top-3 평가
# =========================
sbert_e1, sbert_ek, sbert_top1_acc, sbert_top3_acc = eval_top1_topk(gold, sbert_top1, sbert_top3)
llm_e1, llm_ek, llm_top1_acc, llm_top3_acc = eval_top1_topk(gold, llm_top1, llm_top3)

print("========== (1) SBERT vs LLM: Top-1 / Top-3 ==========")
print(f"[SBERT] Top-1 Accuracy: {sbert_top1_acc:.4f} ({sbert_e1['correct_top1'].sum()}/{len(sbert_e1)})")
print(f"[SBERT] Top-3 Accuracy: {sbert_top3_acc:.4f} ({sbert_ek['hit_topk'].sum()}/{len(sbert_ek)})")
print()
print(f"[LLM ] Top-1 Accuracy: {llm_top1_acc:.4f} ({llm_e1['correct_top1'].sum()}/{len(llm_e1)})")
print(f"[LLM ] Top-3 Accuracy: {llm_top3_acc:.4f} ({llm_ek['hit_topk'].sum()}/{len(llm_ek)})")

# =========================
# 3) McNemar test (Top-1 기준)
# =========================
print("\n========== (2) McNemar test (Top-1): SBERT vs LLM ==========")

# 같은 샘플만 비교
cmp = gold.merge(
    sbert_e1[["job_posting_id", "correct_top1"]].rename(columns={"correct_top1": "sbert_correct"}),
    on="job_posting_id", how="inner"
).merge(
    llm_e1[["job_posting_id", "correct_top1"]].rename(columns={"correct_top1": "llm_correct"}),
    on="job_posting_id", how="inner"
)

a = int(((cmp["sbert_correct"] == True)  & (cmp["llm_correct"] == True)).sum())
b = int(((cmp["sbert_correct"] == True)  & (cmp["llm_correct"] == False)).sum())
c = int(((cmp["sbert_correct"] == False) & (cmp["llm_correct"] == True)).sum())
d = int(((cmp["sbert_correct"] == False) & (cmp["llm_correct"] == False)).sum())

print("Contingency table (Top-1):")
print(f"  a(both correct) = {a}")
print(f"  b(SBERT correct, LLM wrong) = {b}")
print(f"  c(SBERT wrong, LLM correct) = {c}")
print(f"  d(both wrong) = {d}")
print(f"  N = {a+b+c+d}")

p_exact = mcnemar_exact_p(b, c)

# 참고용 chi-square(cc)
if (b + c) == 0:
    chi2_cc = 0.0
else:
    chi2_cc = ((abs(b - c) - 1) ** 2) / (b + c)

print(f"\nMcNemar exact p-value (two-sided): {p_exact:.6f}")
print(f"McNemar chi-square (cc): {chi2_cc:.4f} (df=1)")
print("Decision (alpha=0.05):", "SIGNIFICANT" if p_exact < 0.05 else "NOT significant")

# =========================
# (선택) 결과 저장
# =========================
# sbert_e1.to_csv("outputs/eval_sbert_top1.csv", index=False, encoding="utf-8-sig")
# llm_e1.to_csv("outputs/eval_llm_top1.csv", index=False, encoding="utf-8-sig")
# cmp.to_csv("outputs/mcnemar_compare_sbert_llm.csv", index=False, encoding="utf-8-sig")

========== (1) SBERT vs LLM: Top-1 / Top-3 ==========
[SBERT] Top-1 Accuracy: 0.1500 (3/20)
[SBERT] Top-3 Accuracy: 0.5500 (11/20)

[LLM ] Top-1 Accuracy: 0.1000 (2/20)
[LLM ] Top-3 Accuracy: 0.3000 (6/20)

========== (2) McNemar test (Top-1): SBERT vs LLM ==========
Contingency table (Top-1):
  a(both correct) = 0
  b(SBERT correct, LLM wrong) = 3
  c(SBERT wrong, LLM correct) = 2
  d(both wrong) = 15
  N = 20

McNemar exact p-value (two-sided): 1.000000
McNemar chi-square (cc): 0.0000 (df=1)
Decision (alpha=0.05): NOT significant
